# VGG Varying Lights Ablation Study

This example shows how one can experiment how changing the number of lights being optimized influences the optimization results. Here, the `CarStudioScene` has been configured with three different lighting configurations, as shown in the [demo video](https://youtu.be/px7gxgCySMQ?si=6aJPJJbKIZDlLdwX&t=87).


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from utils.optimize import optimize_with_criterion
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.record_keeping.experiment import FolderManager
from losses.image_image import VGGStyleTransferLoss
from examples.example_scenes import (
    SciFiRobotScene,
    SpringScene,
    CarScene,
    BlenderManScene,
    RedCarScene,
    CandleScene,
    HouseScene,
    DinoScene,
    FlowerPotScene,
    CarStudioScene,
    EinarScene,
    EinarSmallDomeScene,
    SpringPortraitScene,
    SpringPortraitSmallDomeScene,
)


In [ ]:
# Define light rig configurations to benchmark
configurations = ['dome_lights', 'four_small_area_lights', 'single_sun_light']
scenes_to_test = [
    CarStudioScene(configuration=config, device=device) for config in configurations
]

In [ ]:
target_image_path = ""  # TODO: Update with target style reference image path

In [ ]:
# Hyperparameters
lr = 0.06
n_iter = 250
global_seed = 2
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())
save_loss_plot_each_iteration = True

output_directory = "vgg_varying_lights_ablation"
title_prefix = "VGG Style Loss"

for scene, configuration in zip(scenes_to_test, configurations):
    print(f"--- Running VGG Light Ablation on {scene.name} ({configuration}) ---")

    criterion = VGGStyleTransferLoss(reference_image=target_image_path)

    optimize_with_criterion(
        scene,
        lr,
        n_iter,
        criterion,
        starting_multiplier_std=(0.1, 0.1, 0.1),
        output_subdirectory_name=output_directory,
        n_results=1,
        render_color_space_converter=color_space_converter,
        require_physically_plausible_multipliers=True,
        title_prefix=f"{title_prefix} ({configuration})",
        device=device,
        save_every=50,
        model_name="VGG-16",
        seed=global_seed,
        save_loss_plot_each_iteration=save_loss_plot_each_iteration,
    )
